<a href="https://colab.research.google.com/github/lakkshanarams-maker/language-Specification/blob/main/Language_Specifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q tensorflow scikit-learn pandas numpy

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving dataset.csv to dataset (2).csv


In [ ]:
import pandas as pd
import numpy as np
import re

In [ ]:
file_name = next(iter(uploaded))

In [ ]:
df = pd.read_csv(file_name)

In [ ]:
print("First five rows:")
print(df.head())

First five rows:
                                                Text  language
0  klement gottwaldi surnukeha palsameeriti ning ...  Estonian
1  sebes joseph pereira thomas  på eng the jesuit...   Swedish
2  ถนนเจริญกรุง อักษรโรมัน thanon charoen krung เ...      Thai
3  விசாகப்பட்டினம் தமிழ்ச்சங்கத்தை இந்துப் பத்திர...     Tamil
4  de spons behoort tot het geslacht haliclona en...     Dutch


In [ ]:
print("\nColumn names:")
print(df.columns.tolist())


Column names:
['Text', 'language']


In [ ]:
print("\nDataset shape:")
print(df.shape)


Dataset shape:
(22000, 2)


In [ ]:
df = df[["Text", "language"]].copy()

In [ ]:
df.dropna(subset=["Text", "language"], inplace=True)

In [ ]:
df["Text"] = df["Text"].astype(str).str.strip()
df["language"] = df["language"].astype(str).str.strip()

In [ ]:
df = df[
    (df["Text"] != "") &
    (df["language"] != "")
]

In [ ]:
df.drop_duplicates(inplace=True)

In [ ]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"\s+", " ", text)
    return text.strip()

In [ ]:
df["Text"] = df["Text"].apply(clean_text)

In [ ]:
print(df.head())
print("Cleaned dataset shape:", df.shape)

                                                Text  language
0  klement gottwaldi surnukeha palsameeriti ning ...  Estonian
1  sebes joseph pereira thomas på eng the jesuits...   Swedish
2  ถนนเจริญกรุง อักษรโรมัน thanon charoen krung เ...      Thai
3  விசாகப்பட்டினம் தமிழ்ச்சங்கத்தை இந்துப் பத்திர...     Tamil
4  de spons behoort tot het geslacht haliclona en...     Dutch
Cleaned dataset shape: (21858, 2)


In [ ]:
MAX_SAMPLES = 5000
df = df.groupby(
    "language",
    group_keys=False
).sample(
    frac=1,
    random_state=42
)

In [ ]:
samples_per_language = max(
    1, MAX_SAMPLES // df["language"].nunique()
)

In [ ]:
df = (
    df.groupby("language", group_keys=False)
    .head(samples_per_language)
    .reset_index(drop=True)
)

In [ ]:
print("Selected samples:", len(df))
print("\nSamples per language:")
print(df["language"].value_counts())

Selected samples: 4994

Samples per language:
language
Arabic        227
Chinese       227
Dutch         227
English       227
Estonian      227
French        227
Hindi         227
Indonesian    227
Japanese      227
Korean        227
Latin         227
Persian       227
Portugese     227
Pushto        227
Romanian      227
Russian       227
Spanish       227
Swedish       227
Tamil         227
Thai          227
Turkish       227
Urdu          227
Name: count, dtype: int64


In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

In [ ]:
X = df["Text"].values

In [ ]:
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(df["language"])

In [ ]:
num_classes = len(label_encoder.classes_)

In [ ]:
print("Language classes:", label_encoder.classes_)
print("Training labels shape:", y_train.shape)
print("Model output shape:", model.output_shape)

Language classes: ['Arabic' 'Chinese' 'Dutch' 'English' 'Estonian' 'French' 'Hindi'
 'Indonesian' 'Japanese' 'Korean' 'Latin' 'Persian' 'Portugese' 'Pushto'
 'Romanian' 'Russian' 'Spanish' 'Swedish' 'Tamil' 'Thai' 'Turkish' 'Urdu']
Training labels shape: (3995,)
Model output shape: (None, 22)


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [ ]:
print("Number of languages:", num_classes)
print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Number of languages: 22
Training samples: 3995
Testing samples: 999


In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [ ]:
vocab_size = 20000
max_length = 100

In [ ]:
tokenizer = Tokenizer(
    num_words=vocab_size,
    oov_token="<OOV>"
)

In [ ]:
tokenizer.fit_on_texts(X_train)

In [ ]:
train_sequences = tokenizer.texts_to_sequences(X_train)
test_sequences = tokenizer.texts_to_sequences(X_test)

In [ ]:
X_train_pad = pad_sequences(
    train_sequences,
    maxlen=max_length,
    padding="post",
    truncating="post"
)

In [ ]:
X_test_pad = pad_sequences(
    test_sequences,
    maxlen=max_length,
    padding="post",
    truncating="post"
)

In [ ]:
actual_vocab_size = min(
    vocab_size,
    len(tokenizer.word_index) + 1
)

In [ ]:
print("Training sequence shape:", X_train_pad.shape)
print("Testing sequence shape:", X_test_pad.shape)

Training sequence shape: (3995, 100)
Testing sequence shape: (999, 100)


In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

In [ ]:
model = Sequential([
    Embedding(
        input_dim=actual_vocab_size,
        output_dim=64,
        mask_zero=True
    ),
    LSTM(64),
    Dropout(0.3),
    Dense(32, activation="relu"),
    Dropout(0.2),
    Dense(num_classes, activation="softmax")
])

In [ ]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
model.build(input_shape=(None, max_length))
model.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 100, 64)        │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 64)             │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 22)             │           726 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,315,830 (5.02 MB)

 Trainable params: 1,315,830 (5.02 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

In [ ]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True
)

In [ ]:
history = model.fit(
    X_train_pad,
    y_train,
    epochs=10,
    batch_size=64,
    validation_split=0.1,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/10
57/57 ━━━━━━━━━━━━━━━━━━━━ 9s 103ms/step - accuracy: 0.0773 - loss: 2.9933 - val_accuracy: 0.1150 - val_loss: 2.6963
Epoch 2/10
57/57 ━━━━━━━━━━━━━━━━━━━━ 7s 124ms/step - accuracy: 0.1797 - loss: 2.5585 - val_accuracy: 0.2200 - val_loss: 2.4450
Epoch 3/10
57/57 ━━━━━━━━━━━━━━━━━━━━ 10s 117ms/step - accuracy: 0.3213 - loss: 2.0877 - val_accuracy: 0.5100 - val_loss: 1.6957
Epoch 4/10
57/57 ━━━━━━━━━━━━━━━━━━━━ 9s 96ms/step - accuracy: 0.5346 - loss: 1.4647 - val_accuracy: 0.7075 - val_loss: 1.1171
Epoch 5/10
57/57 ━━━━━━━━━━━━━━━━━━━━ 7s 125ms/step - accuracy: 0.6776 - loss: 1.0843 - val_accuracy: 0.8250 - val_loss: 0.8575
Epoch 6/10
57/57 ━━━━━━━━━━━━━━━━━━━━ 10s 118ms/step - accuracy: 0.7864 - loss: 0.8211 - val_accuracy: 0.7975 - val_loss: 0.7869
Epoch 7/10
57/57 ━━━━━━━━━━━━━━━━━━━━ 6s 96ms/step - accuracy: 0.8209 - loss: 0.7135 - val_accuracy: 0.7125 - val_loss: 1.0344
Epoch 8/10
57/57 ━━━━━━━━━━━━━━━━━━━━ 5s 95ms/step - accuracy: 0.7777 - loss: 0.7621 - val_accuracy: 0.7

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)
import numpy as np

In [ ]:
y_prob = model.predict(X_test_pad)
y_pred = np.argmax(y_prob, axis=1)

32/32 ━━━━━━━━━━━━━━━━━━━━ 1s 28ms/step


In [ ]:
print("Accuracy:", round(
    accuracy_score(y_test, y_pred), 4
))

print("Precision:", round(
    precision_score(
        y_test, y_pred,
        average="macro", zero_division=0
    ), 4
))

print("Recall:", round(
    recall_score(
        y_test, y_pred,
        average="macro", zero_division=0
    ), 4
))

print("F1-score:", round(
    f1_score(
        y_test, y_pred,
        average="macro", zero_division=0
    ), 4
))


Accuracy: 0.8478
Precision: 0.8437
Recall: 0.8473
F1-score: 0.8367


In [ ]:
print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    target_names=label_encoder.classes_,
    zero_division=0
))


Classification Report:
              precision    recall  f1-score   support

      Arabic       0.86      0.84      0.85        45
     Chinese       0.45      0.91      0.60        45
       Dutch       1.00      0.87      0.93        46
     English       0.80      0.91      0.85        45
    Estonian       0.66      0.91      0.77        45
      French       0.95      0.93      0.94        45
       Hindi       1.00      0.96      0.98        46
  Indonesian       1.00      0.85      0.92        46
    Japanese       0.00      0.00      0.00        45
      Korean       0.91      0.89      0.90        46
       Latin       0.81      0.93      0.87        46
     Persian       1.00      0.98      0.99        45
   Portugese       1.00      0.76      0.86        45
      Pushto       1.00      0.71      0.83        45
    Romanian       1.00      0.89      0.94        46
     Russian       0.91      0.93      0.92        45
     Spanish       0.94      1.00      0.97        46
   

In [ ]:
import numpy as np
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [ ]:
def predict_language(text):
    text = clean_text(text)

In [ ]:
sequence = tokenizer.texts_to_sequences([text])

In [ ]:
if not sequence or len(sequence[0]) == 0:
    print("Error: The tokenizer does not recognize any words.")
else:
    # Pad the sequence
    padded = pad_sequences(
        sequence,
        maxlen=max_length,
        padding="post",
        truncating="post"
    )

In [ ]:
padded = pad_sequences(
    sequence,
    maxlen=max_length,
    padding="post",
    truncating="post"
)

In [ ]:
prediction = model.predict(padded, verbose=0)


In [ ]:
index = int(np.argmax(prediction[0]))

In [ ]:
language = label_encoder.inverse_transform([index])[0]

In [ ]:
confidence = float(prediction[0][index]) * 100

In [ ]:
print("All prediction scores:", prediction[0])
print("Class labels:", label_encoder.classes_)

All prediction scores: [0.023077   0.21328631 0.01413335 0.01343447 0.07254811 0.00815523
 0.01642088 0.01989606 0.20225763 0.04842956 0.06463448 0.0042325
 0.00833937 0.01210407 0.06808573 0.02104108 0.00445182 0.00607203
 0.04726148 0.1012949  0.02538828 0.00545559]
Class labels: ['Arabic' 'Chinese' 'Dutch' 'English' 'Estonian' 'French' 'Hindi'
 'Indonesian' 'Japanese' 'Korean' 'Latin' 'Persian' 'Portugese' 'Pushto'
 'Romanian' 'Russian' 'Spanish' 'Swedish' 'Tamil' 'Thai' 'Turkish' 'Urdu']


In [ ]:
print(df["language"].value_counts())
print(df[df["language"].str.lower() == "english"].head(10))
print(label_encoder.classes_)

language
Arabic        227
Chinese       227
Dutch         227
English       227
Estonian      227
French        227
Hindi         227
Indonesian    227
Japanese      227
Korean        227
Latin         227
Persian       227
Portugese     227
Pushto        227
Romanian      227
Russian       227
Spanish       227
Swedish       227
Tamil         227
Thai          227
Turkish       227
Urdu          227
Name: count, dtype: int64
                                                  Text language
681  in following a increase in crime the united st...  English
682  the reno news review rnr was founded in when n...  English
683  the age distribution of the population as of i...  English
684  the russon mill a water mill is located in the...  English
685  glaser was educated at st edmund hall oxford g...  English
686  holly can contain caffeic acid caffeoyl deriva...  English
687  linus yale jr improved upon his fathers lock i...  English
688  after a short break reynolds toured again with...  E

In [ ]:
print("English in training data:",
      sum(label_encoder.inverse_transform(y_train) == "English"))

print("English in testing data:",
      sum(label_encoder.inverse_transform(y_test) == "English"))

English in training data: 182
English in testing data: 45


In [ ]:
history = model.fit(
    X_train_pad,
    y_train,
    epochs=10,
    batch_size=64,
    validation_split=0.1,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/10
57/57 ━━━━━━━━━━━━━━━━━━━━ 7s 119ms/step - accuracy: 0.9280 - loss: 0.2010 - val_accuracy: 0.8900 - val_loss: 0.3730
Epoch 2/10
57/57 ━━━━━━━━━━━━━━━━━━━━ 19s 279ms/step - accuracy: 0.9385 - loss: 0.1743 - val_accuracy: 0.9000 - val_loss: 0.3219
Epoch 3/10
57/57 ━━━━━━━━━━━━━━━━━━━━ 12s 125ms/step - accuracy: 0.9435 - loss: 0.1647 - val_accuracy: 0.8800 - val_loss: 0.4304
Epoch 4/10
57/57 ━━━━━━━━━━━━━━━━━━━━ 5s 96ms/step - accuracy: 0.9007 - loss: 0.3229 - val_accuracy: 0.8650 - val_loss: 0.5380


In [ ]:
text = "hi"

In [ ]:
print("Input text:", text)
print("Predicted language:", language)
print(f"Confidence: {confidence:.2f}%")

Input text: hi
Predicted language: Chinese
Confidence: 21.33%
